UWAGA - w tej wersji, **nie ma inputu**, ponieważ pomyślałem, że nie ma sensu go brać do `bedtools multicov`, później doczytałem, że mamy go uwzględnić. Kod jest w stanie go przetworzyć, ale nie ma go w countach, rano będzie wersja która go uwzględnia

Exercise 1

Prepare a count matrix for your BAM files – the features are IDR-passed merged peaks.
1. Get the first three columns from the narrowPeak files (IDR output) and save them as
BED files (cut).
2. Concatenate them (cat).
3. Use bedtools merge to merge overlapping features.
4. Use bedtools multicov to count reads from the BAM files overlapping the features in
the BED file – the output should be a matrix.
5. Add a header to name the columns – for example, you can use echo -e
"colname1\tcolname2…" > header.txt, then concatenate with >> (cat).

In [ ]:
#!/bin/bash
# Script for creating matrixes 

# paths
peak_folder=~/labs5_all/macs2_files
bams_folder=~/labs5_all/base_files
matrix_folder=~/labs5_all/count_matrixes
# put prefix of experiments 
groups_names='Control Sucrose Torin1'

# combining_files
## creating .bed files
for f in $peak_folder/*_peaks.narrowPeak; do
    cut -f1-3 "$f" > "${f%.narrowPeak}.bed"
done
## joining replicates
for condition in $groups_names; do
    cat $peak_folder/${condition}_rep*_peaks.bed > $peak_folder/${condition}_all.bed 
done 
## joining all conditions
cat $peak_folder/*_all.bed > $peak_folder/all_peaks.bed 

# finding features set non overlapping regions
## sorting 
bedtools sort -i $peak_folder/all_peaks.bed > $peak_folder/all_peaks.sorted.bed
## getting rid off overlapping regions 
bedtools merge -i $peak_folder/all_peaks.sorted.bed > $peak_folder/merged_peaks.bed 

# matrixes
## creating matrixes
mkdir -p $matrix_folder
bedtools multicov -bams $bams_folder/*.bam -bed $peak_folder/merged_peaks.bed > $matrix_folder/count_matrix.tsv

# UWAGA - potrzeba mieć pliki 
# for bam in *.bam; do samtools index $bam; done

# header
## creating header 
header="chrom\tstart\tend"
for bam in $bams_folder/*.bam; do
    sample=$(basename "$bam" .bam)
    header="$header\t$sample"
done
echo -e "$header" > header.txt
## joining files
cat header.txt $matrix_folder/count_matrix.tsv > $matrix_folder/count_matrix_named.tsv
rm header.txt

Exercise 2

Import the matrix into R and perform PCA.
1. Correct the matrix if needed – column names should be sample names, and row
names should be feature names (these can simply be “peak” followed by a number).
2. Create a colData object containing the condition factor.
3. Create a DESeq2 dataset from the matrix.
4. Apply VST (variance stabilizing transformation).
5. Generate a PCA plot and save it – use all peaks (not just the top 500), and color
points by condition.
6. Remove the “Input” sample from the VST matrix and generate another PCA plot,
then save it.


In [ ]:
# loading libraries
options(encoding = "UTF-8")
Sys.setlocale("LC_ALL", "pl_PL.UTF-8")

# packages loader
use_package <- function(pkg) {
  if (!require(pkg, character.only = TRUE)) {
    install.packages(pkg)
    library(pkg, character.only = TRUE)
  }
}

use_package('DESeq2')
use_package('ggplot2')

In [ ]:
# loading data
counts_data = read.table('count_matrix_named.tsv', header=TRUE, row.names=NULL)
# get rid of chrom, start, end information
counts_data = counts_data[, -(1:3)]
# create row names
rownames(counts_data) = paste0('peak', seq_len(nrow(counts_data))) 
head(counts_data)

In [ ]:
# creating DESeq object
## create colData object 
sample_names = colnames(counts_data)
condition = factor(sub("_.*", "", sample_names))
colData = data.frame(row.names=sample_names, condition=condition)

## create DeSeqObject
dds = DESeqDataSetFromMatrix(countData = counts_data, 
                            colData=colData, 
                            design = ~condition)
head(dds)

In [ ]:
# transforming variance
vsd = vst(dds)
head(vsd)

In [ ]:
# plots
plt = plotPCA(vsd, intgroup = "condition")
ggsave("PCA_all_samples.png", plot=plt)

# display
plt

In [ ]:
# plots without `input`
# removing col
vsd_no_input <- vsd[, colData$condition != "Input.sorted.blacklist.mapq20"]

# creating plot 
plt = plotPCA(vsd_no_input, intgroup = "condition")
ggsave("PCA_no_input.png", plot=plt)

# display
plt